In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_sample_weight

from src.common.load_data import load_data, drop_targets, get_test_sets
from src.common.treat_missing import load_and_process_for_random_forest
from src.common.evaluations import evaluate_classification
from src.utils.plots import plot_loss_curve, plot_roc_auc_curve, plot_precision_recall_curve

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
warnings.filterwarnings("ignore")
pd.options.display.max_rows = 200


**How to fix overfitting in XGBoost?**

| If your model is overfitting | Action to take on Hyperparameter |
|---|---|
| max_depth | Decrease it (e.g., from 8 to 3-4) |
| min_child_weight | Increase it (e.g., from 1 to 5-10) |
| gamma | Increase it (e.g., from 0 to 0.2-1.0) - requires a bigger loss reduction before splitting |
| reg_alpha (L1) | Increase it (e.g., from 0 to 0.1-1.0) |
| reg_lambda (L2) | Increase it (e.g., from 1.0 to 5-20) |
| subsample | Decrease it (e.g., from 1.0 to 0.6-0.8) |
| colsample_bytree | Decrease it (e.g., from 1.0 to 0.6-0.8) |
| learning_rate | Decrease it, and increase n_estimators to compensate |


## 1. Load Data

In [ ]:
df = load_data("./data/cibil_score/cibil_score.csv")
X, y_lin_reg, y_binary, y_multiclass = drop_targets(df)
X_train, X_test, y_train, y_test = get_test_sets(X, y_multiclass, test_size=0.3)


## 2. Preprocessing

In [ ]:
X_train_processed, X_test_processed = load_and_process_for_random_forest(X_train, X_test)


## 3. XGBoost Multiclass Classifier

In [ ]:
xgb_multiclass = Pipeline(
    steps=[
        ("model", XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ))
    ]
)

xgb_multiclass.fit(
    X_train_processed,
    y_train,
)


## 4. XGBoost Multiclass Classifier with sample_weight "balanced"

In [ ]:
# XGBoost has no built-in `class_weight` argument like RandomForestClassifier.
# The equivalent is passing per-sample weights at fit time; `compute_sample_weight`
# turns a class-weighting scheme (here "balanced") into one weight per row.
sample_weight_balanced = compute_sample_weight(class_weight="balanced", y=y_train)

xgb_multiclass1 = Pipeline(
    steps=[
        ("model", XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ))
    ]
)

xgb_multiclass1.fit(
    X_train_processed,
    y_train,
    model__sample_weight=sample_weight_balanced,
)


## 5. XGBoost Multiclass Classifier with custom sample_weight

In [ ]:
class_weight = {
    "P1": 1.0,
    "P2": 0.5,
    "P3": 2.0,
    "P4": 1.0
}

sample_weight_custom = compute_sample_weight(class_weight=class_weight, y=y_train)

xgb_multiclass2 = Pipeline(
    steps=[
        ("model", XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ))
    ]
)

xgb_multiclass2.fit(
    X_train_processed,
    y_train,
    model__sample_weight=sample_weight_custom,
)


## 6. Evaluate all three variants of Multiclass Models

In [ ]:
multiclass_results = evaluate_classification(
    xgb_multiclass,
    X_test_processed,
    y_test,
    binary=False
)

print("\n==============================")
print("XGBoost Multiclass")
print("==============================")

for metric, value in multiclass_results.items():
    print(f"{metric}: {value}")

multiclass_results1 = evaluate_classification(
    xgb_multiclass1,
    X_test_processed,
    y_test,
    binary=False
)

print("\n===================================================")
print("XGBoost Multiclass sample_weight=\'balanced\'")
print("===================================================")

for metric, value in multiclass_results1.items():
    print(f"{metric}: {value}")

multiclass_results2 = evaluate_classification(
    xgb_multiclass2,
    X_test_processed,
    y_test,
    binary=False
)

print("\n==============================================")
print("XGBoost Multiclass custom sample_weight")
print("==============================================")

for metric, value in multiclass_results2.items():
    print(f"{metric}: {value}")


*(Fill in after running section 6 - e.g. the same Macro F1 / P3 Recall comparison table you used for the Random Forest notebook. This tells you which weighting scheme to carry forward into the random search below; the search is currently set up assuming the custom-weighted variant wins, same as it did for Random Forest - adjust section 7 if your results favor a different variant.)*

## 7. Random Search for Best Parameters

In [ ]:
param_distributions = {
    "model__n_estimators": [100, 200, 300, 400],
    "model__max_depth": [3, 4, 5, 6, 8],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "model__subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__min_child_weight": [1, 3, 5, 7, 10],
    "model__gamma": [0, 0.1, 0.2, 0.5, 1.0],
    "model__reg_alpha": [0, 0.001, 0.01, 0.1, 1.0],
    "model__reg_lambda": [0.1, 1.0, 5.0, 10.0, 20.0],
}

# IMPORTANT: n_jobs is left off the inner XGBClassifier here on purpose.
# RandomizedSearchCV below already parallelizes across candidates with
# n_jobs=-1; if the inner estimator ALSO grabs every core, you get far more
# worker processes than CPU cores, which causes massive slowdowns instead
# of speedups (this is exactly what happened with the Random Forest search).
xgb_search_base = Pipeline(
    steps=[
        ("model", XGBClassifier(
            n_jobs=1,
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
        ))
    ]
)

xgb_random_search = RandomizedSearchCV(
    estimator=xgb_search_base,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="f1_macro",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    verbose=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

# sample_weight is indexable by row, so sklearn automatically slices it to
# match each CV fold's train indices - no extra work needed here.
xgb_random_search.fit(
    X_train_processed,
    y_train,
    model__sample_weight=sample_weight_custom,
)

print("Best Parameters:")
for param, value in xgb_random_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV Score (Macro F1): {xgb_random_search.best_score_:.4f}")


## 8. Run Classification Using Best Parameters

In [ ]:
# Strip the pipeline step prefix ("model__") so the params can be passed
# straight into a fresh XGBClassifier
best_params = {
    k.replace("model__", ""): v for k, v in xgb_random_search.best_params_.items()
}

xgb_best = Pipeline(
    steps=[
        ("model", XGBClassifier(
            **best_params,
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ))
    ]
)

xgb_best.fit(
    X_train_processed,
    y_train,
    model__sample_weight=sample_weight_custom,
)


## 9. Add Regularization - Model with best parameters

In [ ]:
# The random search above already tunes reg_alpha, reg_lambda, gamma,
# subsample and colsample_bytree alongside the other hyperparameters, so
# `best_params` may already include reasonable regularization. This cell
# pushes those specific knobs further in the "reduce overfitting" direction
# (see the reference table at the top of the notebook) and compares
# train vs. test accuracy before/after, so you can see whether the extra
# regularization actually closes the gap for THIS dataset.
regularized_params = dict(best_params)
regularized_params["reg_alpha"] = max(best_params.get("reg_alpha", 0), 0.1) * 5
regularized_params["reg_lambda"] = max(best_params.get("reg_lambda", 1.0), 1.0) * 2
regularized_params["gamma"] = best_params.get("gamma", 0) + 0.3
regularized_params["subsample"] = min(best_params.get("subsample", 0.8), 0.7)
regularized_params["colsample_bytree"] = min(best_params.get("colsample_bytree", 0.8), 0.7)

xgb_regularized = Pipeline(
    steps=[
        ("model", XGBClassifier(
            **regularized_params,
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ))
    ]
)

xgb_regularized.fit(
    X_train_processed,
    y_train,
    model__sample_weight=sample_weight_custom,
)

print("Regularized parameters:")
for param, value in regularized_params.items():
    print(f"  {param}: {value}")

print("\nTrain vs. test accuracy, before vs. after extra regularization:")
for name, model in [
    ("Best Params (from search)", xgb_best),
    ("Best Params + Extra Regularization", xgb_regularized),
]:
    train_score = model.score(X_train_processed, y_train)
    test_score = model.score(X_test_processed, y_test)
    print(f"  {name}: train={train_score:.4f}  test={test_score:.4f}  gap={train_score - test_score:.4f}")

# Carry the regularized model forward as the "final" model for the rest of
# the notebook. If the extra regularization actually hurt test performance,
# swap this back to xgb_best instead.
xgb_final = xgb_regularized


## 10. Plot Train and Test Loss Curve - Model with best parameters

In [ ]:
# Unlike Random Forest, XGBoost is a boosting model with a genuine
# iteration-by-iteration loss: passing `eval_set` lets XGBoost track the
# metric after each boosting round on both train and test, no warm-start
# hack needed. We rebuild the model outside the Pipeline just so we can
# pass `eval_set`/`verbose` directly to `.fit()`.
xgb_curve_model = XGBClassifier(
    **regularized_params,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

xgb_curve_model.fit(
    X_train_processed,
    y_train,
    sample_weight=sample_weight_custom,
    eval_set=[(X_train_processed, y_train), (X_test_processed, y_test)],
    verbose=False,
)

eval_results = xgb_curve_model.evals_result()
train_losses = eval_results["validation_0"]["mlogloss"]
test_losses = eval_results["validation_1"]["mlogloss"]

plot_loss_curve(
    train_losses,
    test_losses,
    title="Train vs Test Log Loss - XGBoost Multiclass (Best Parameters + Regularization)",
)


## 11. Plot ROC-AUC Curve - Model with best parameters

In [ ]:
class_labels = xgb_final.named_steps["model"].classes_.tolist()
y_test_scores = xgb_final.predict_proba(X_test_processed)

plot_roc_auc_curve(
    y_test,
    y_test_scores,
    classes=class_labels,
    title="ROC Curve (One-vs-Rest) - XGBoost Multiclass (Best Parameters + Regularization)",
)


## 12. Plot Precision-Recall Curve - Model with best parameters

In [ ]:
plot_precision_recall_curve(
    y_test,
    y_test_scores,
    classes=class_labels,
    title="Precision-Recall Curve (One-vs-Rest) - XGBoost Multiclass (Best Parameters + Regularization)",
)


## 13. Evaluate Multiclass Classification - Model with best parameters

In [ ]:
multiclass_results_best = evaluate_classification(
    xgb_final,
    X_test_processed,
    y_test,
    binary=False
)

print("\n==============================")
print("XGBoost Multiclass - Best Parameters + Regularization")
print("==============================")

for metric, value in multiclass_results_best.items():
    print(f"{metric}: {value}")


## 14. Feature Importance

In [ ]:
if hasattr(X_train_processed, "columns"):
    feature_names = list(X_train_processed.columns)
else:
    feature_names = [f"feature_{i}" for i in range(X_train_processed.shape[1])]

# Built-in gain-based importances
importances = xgb_final.named_steps["model"].feature_importances_

feature_importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Top 20 features by built-in (gain) importance:")
print(feature_importance_df.head(20))

top_n = min(20, len(feature_importance_df))
top_features = feature_importance_df.head(top_n)

plt.figure(figsize=(8, 8))
plt.barh(top_features["feature"][::-1], top_features["importance"][::-1])
plt.xlabel("Feature Importance (gain)")
plt.title(f"Top {top_n} Feature Importances - XGBoost (Best Parameters + Regularization)")
plt.tight_layout()
plt.show()

# Permutation importance - measures the actual drop in held-out performance,
# so it's less biased toward high-cardinality features than gain-based importance
perm_result = permutation_importance(
    xgb_final,
    X_test_processed,
    y_test,
    scoring="f1_macro",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

perm_importance_df = (
    pd.DataFrame({
        "feature": feature_names,
        "importance_mean": perm_result.importances_mean,
        "importance_std": perm_result.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

print("\nTop 20 features by permutation importance:")
print(perm_importance_df.head(20))

plt.figure(figsize=(8, 8))
top_perm = perm_importance_df.head(top_n)
plt.barh(top_perm["feature"][::-1], top_perm["importance_mean"][::-1],
         xerr=top_perm["importance_std"][::-1])
plt.xlabel("Permutation Importance (mean decrease in Macro F1)")
plt.title(f"Top {top_n} Permutation Importances - XGBoost (Best Parameters + Regularization)")
plt.tight_layout()
plt.show()

# Keep the features that matter - here, anything with above-average
# permutation importance. Adjust the threshold/count to taste.
important_features = perm_importance_df.loc[
    perm_importance_df["importance_mean"] > perm_importance_df["importance_mean"].mean(),
    "feature"
].tolist()

print(f"\nSelected {len(important_features)} important features out of {len(feature_names)}:")
print(important_features)


## 15. Evaluate Classification with Important Features and Best Parameters

In [ ]:
# Make sure we're working with DataFrames so we can subset by column name,
# even if the preprocessing step returned a plain numpy array
X_train_df = (
    X_train_processed if isinstance(X_train_processed, pd.DataFrame)
    else pd.DataFrame(X_train_processed, columns=feature_names)
)
X_test_df = (
    X_test_processed if isinstance(X_test_processed, pd.DataFrame)
    else pd.DataFrame(X_test_processed, columns=feature_names)
)

X_train_important = X_train_df[important_features]
X_test_important = X_test_df[important_features]

xgb_final_important = Pipeline(
    steps=[
        ("model", XGBClassifier(
            **regularized_params,
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ))
    ]
)

# sample_weight_custom is one weight per row, so it's unaffected by
# dropping columns and can be reused as-is.
xgb_final_important.fit(
    X_train_important,
    y_train,
    model__sample_weight=sample_weight_custom,
)

multiclass_results_important = evaluate_classification(
    xgb_final_important,
    X_test_important,
    y_test,
    binary=False
)

print("\n==============================")
print("XGBoost Multiclass - Best Parameters + Regularization + Important Features")
print("==============================")

for metric, value in multiclass_results_important.items():
    print(f"{metric}: {value}")
